# **Домашнее задание 1: Оптический поток и динамика видео**

### **Цель**

Освоить практическое применение оптического потока для анализа движения, научиться строить полные видеопайплайны, включающие вычисление потока, фильтрацию, warping и стабилизацию. Студент выбирает **одну** из двух инженерных задач: построение мини-системы **стабилизации камеры** или создание **трекинг-модуля движения**, анализирующего траектории и устойчивость методов Lucas–Kanade и Farnebäck.

---

# **Вариант A: Мини-система стабилизации камеры**

### **Задание**

1. Выбрать видеоролик с заметной дрожью камеры (телефонная съёмка, action-камера, GoPro, записанная вручную панорама).
2. Вычислить плотный оптический поток (Farnebäck) или разреженный (LK) и оценить глобальное движение камеры по кадрам (аффинная модель или гомография).
3. Сгладить траекторию движения низкочастотным фильтром (скользящее окно, экспоненциальное сглаживание или фильтр Калмана по желанию).
4. Выполнить **компенсацию движения** через warp каждого кадра к стабилизированной траектории.
5. Сформировать стабилизированное видео.
6. Построить визуальные сравнения «до/после» и провести **error analysis**: какие участки стабилизируются плохо и почему (оптический поток, тени, motion blur, нехватка текстуры, ошибки глобальной модели).

---

# **Вариант B: Учебная система анализа движения (трекинг-модуль)**

### **Задание**

1. Выбрать любой видеоролик с несколькими движущимися объектами.
2. Реализовать вычисление разреженного оптического потока (Lucas–Kanade) и плотного (Farnebäck).
3. Для LK:
   – найти ключевые точки;
   – отследить траектории по 50–200 кадрам;
   – визуализировать движение в координатной системе кадра.
4. Для Farnebäck:
   – построить поле движения;
   – выделить движущиеся объекты через модуль потока;
   – получить бинарные маски движения.
5. Сравнить чувствительность методов к текстуре, motion blur, теням, шуму.
6. Провести **качественный и количественный разбор ошибок**: где LK теряет точки, где плотный поток выдаёт шум, где маска движения фрагментируется.
7. Подготовить краткое исследование: по каким признакам студент выбирал бы LK или Farnebäck в реальной задаче.

---

# **Требования к сдаче**

Репозиторий на GitHub:

* `src/` или ноутбук с кодом вычисления оптического потока и всего пайплайна.
* `README.md` c описанием выбранного варианта (A или B), параметров, применённых фильтров и основных наблюдений.
* Визуализации:
  – для варианта A: примерные кадры «до/после», графики траектории движения камеры, примеры warping-а;
  – для варианта B: траектории LK, карты плотного потока, маски движения.
* При варианте A — итоговое стабилизированное видео.
* При варианте B — сравнительный отчёт о различиях LK и Farnebäck.

---

# **Критерии оценки**

| Баллы | Критерий                                                                      |
| ----- | ----------------------------------------------------------------------------- |
| 0–3   | Полнота: реализованы все этапы выбранного варианта (A или B).                 |
| 0–3   | Код: чистый, воспроизводимый, корректно использует OpenCV.                    |
| 0–2   | Анализ: качественный разбор ошибок, интерпретация результатов, выводы.        |
| 0–2   | Репозиторий: аккуратность, читаемость, примеры визуализаций, понятный README. |

**Максимум: 10 баллов.**

---


In [44]:
import cv2
import numpy as np

cap = cv2.VideoCapture("scube.mp4")
feature_params = dict(maxCorners=400,
                      qualityLevel=0.1,
                      minDistance=3,
                      blockSize=7)
lk_params = dict(winSize=(51, 51),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 15, 0.03))
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
mask = np.zeros_like(old_frame)
good_new = None 
for i in range(200):
    ret, frame = cap.read()
    if not ret:
        break
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if p0 is not None:
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]
            for new, old in zip(good_new, good_old):
                a, b = new.ravel()
                c, d = old.ravel()
                mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), (0,255,0), 2)
                frame = cv2.circle(frame, (int(a), int(b)), 5, (0,0,255), -1)
    if good_new is None or len(good_new) < 50 or i == 28:
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
    else:
        p0 = good_new.reshape(-1, 1, 2)
    img = cv2.add(frame, mask)
    cv2.imshow("LK flow", img)
    old_gray = frame_gray.copy()
    if cv2.waitKey(30) & 0xFF == 27:
        break
cap.release()
cv2.destroyAllWindows()

In [17]:
cap = cv2.VideoCapture("scube.mp4")
ret, frame1 = cap.read()
prev = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

while True:
    ret, frame2 = cap.read()
    if not ret:
        break
    next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    # prev = cv2.GaussianBlur(prev, (5,5), 0)
    # next = cv2.GaussianBlur(next, (5,5), 0)
    flow = cv2.calcOpticalFlowFarneback(
        prev, next, None,
        pyr_scale=0.5,
        levels=4,
        winsize=25,
        iterations=5,
        poly_n=7,
        poly_sigma=1.5,
        flags=cv2.OPTFLOW_FARNEBACK_GAUSSIAN
    )
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(frame2)
    hsv[..., 1] = 255
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    _, motion_mask = cv2.threshold(mag, 2.0, 255, cv2.THRESH_BINARY)
    # kernel = np.ones((5,5), np.uint8)
    # motion_mask = cv2.morphologyEx(motion_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
    # motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_DILATE, kernel)
    cv2.imshow("dense", rgb)
    cv2.imshow("mask", motion_mask.astype(np.uint8))
    prev = next

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()